# D1：PyBaMM 电池仿真 —— 充放电特性
# D1: PyBaMM Battery Simulation — Charge/Discharge

## 一、实验原理 | Principle
### 工程背景 | Engineering Background

走进任何一家电池厂或车企的电池实验室，你都会看到排成阵列的充放电柜：电芯在里面以不同倍率反复充放电，工程师记录每一条电压曲线，据此定容量、分档次、判寿命。**仿真（PyBaMM）把这间实验室搬进了电脑**：电芯样品还没做出来，就能预知它的放电曲线长什么样——选错材料体系或电极设计，代价是数亿元的产线投资。本实验用行业通用的 Chen2020 参数集（真实商用 NMC 电芯的测量参数）跑一条 1C 放电曲线，就是电芯工程师每天在看的东西。

### 1.1 为什么仿真电池？
电池是电动车最贵的部件。设计电池包前，需要预测：
- 不同倍率放电下，电压掉多快？（影响动力）
- 能放出多少容量？（影响续航）
- 快充会不会析锂？（影响寿命和安全）

### 1.2 PyBaMM
**PyBaMM**（剑桥、牛津等开发）：基于**电化学偏微分方程**的电池仿真，
不是简单的等效电路拟合。**SPM**（单颗粒模型）把每个电极简化为一个球形颗粒，
计算快、适合教学。**Chen2020** 是商用 NMC 电池的实测参数。

### 1.3 关键概念
- **C-rate**：充放电倍率。1C = 1小时放完，2C = 半小时放完
- **OCV**：开路电压（静置时的电压）；**SOC**：剩余电量百分比
- **极化**：大电流放电时电压的额外下降（内阻效应）


### 1.4 SPM 模型详解 | Single Particle Model

PyBaMM 的 SPM（单颗粒模型）假设：
- 每个电极用**一个代表性颗粒**近似（忽略电极厚度方向的不均匀性）
- 锂离子在颗粒内的扩散用 Fick 定律描述
- 电解液浓度均匀（忽略浓差极化）

**控制方程**：
$$ \frac{\partial c_s}{\partial t} = \frac{1}{r^2}\frac{\partial}{\partial r}\left(D_s r^2 \frac{\partial c_s}{\partial r}\right) $$

**SPM vs DFN**：
| 特征 | SPM | DFN |
|---|---|---|
| 电极厚度方向 | 忽略 | 考虑 |
| 电解液 | 均匀假设 | Nernst-Planck |
| 计算速度 | 快（<1s） | 慢（需粗网格） |
| 适用 | 容量/OCV | 析锂/快充（D4） |

### 1.5 Chen2020 参数集 | Chen2020 Parameter Set

本实验采用 Chen et al. (2020) 的 NMC811/石墨体系参数：
- 标称容量：4.85 Ah（软包单体）
- 电压范围：2.5–4.2 V
- 颗粒半径：负极 5.86 μm，正极 5.22 μm

该参数集是 PyBaMM 社区验证最充分的锂离子参数之一，适合教学。

### 📖 术语速览 | Key Terms

| 术语 Term | 一句话解释 |
|---|---|
| C 率 C-rate | 充放电电流相对于容量的大小：1C = 1 小时放完的电流（本电芯 1C = 5 A） |
| 端电压 Terminal Voltage | 电芯两极实测电压，工作时低于开路电压（被极化"吃掉"一部分） |
| 电压平台 Voltage Plateau | 放电曲线中段近乎平坦的区域，锂电芯大部分容量在这里放出 |
| 极化 Polarization | 电流流过时电压偏离开路值的现象，电流越大偏离越大 |
| SOC | 荷电状态，即"还剩几成电"，放电时从 100% 降到 0% |
| SPM 单粒子模型 | 用一颗正极颗粒+一颗负极颗粒代表整片电极的简化电化学模型 |
## 二、实验目的 | Objectives
1. 掌握 PyBaMM 的 SPM 模型和 Chen2020 参数集
2. 仿真 1C 放电，观察电压/容量曲线
3. 导出 OCV-SOC 数据（供 C1 的 FASTSim 用——这就是 D→C 的链路！）

## 三、学生任务 | Your Tasks
1. **运行** 1C 放电仿真，记录放出的容量（Ah）
2. **对比**：0.5C 和 2C 放电，容量有什么变化？为什么？（提示：极化）
3. **导出**：用 `BatteryFASTSimAdapter.export_from_pybamm()` 生成 CSV
4. **思考题**：快充（3C）时，负极表面会发生什么？（提示：析锂，查 D4）

### 🔬 试一试 | Try It

在代码里加一行改参数（`param['Current function [A]'] = ...` 需放在建 Simulation 之前），先猜后验：

| 改什么 | 怎么改 | 预期效果（先猜后验） |
|---|---|---|
| 倍率 1C → 2C | 电流 5 A → 10 A（仿真时长改 [0, 1800]） | 平台电压更低、放出容量变少、提前到截止 |
| 倍率 1C → 0.5C | 电流 5 A → 2.5 A（时长 [0, 7200]） | 容量更接近 5.0 Ah，曲线更"饱满" |
| 温度 25°C → −10°C | `param['Ambient temperature [K]'] = 263.15` | 平台下移、容量明显下降——冬天续航缩水的微观原因 |
## 四、验证内容 | Verification
- [ ] PyBaMM 仿真收敛（无求解器报错）
- [ ] 放电容量误差 ≤5%（vs 标称 4.85Ah）
- [ ] 成功导出 OCV-SOC CSV 文件
- [ ] CSV 可被 `BatteryFASTSimAdapter` 正确读取

### 常见报错说明 | Troubleshooting
| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `Solver failed` | 求解器不收敛 | 降低 C-rate，或换 SPM 模型 |
| 容量远小于标称 | 参数或倍率设置错误 | 检查 `ParameterValues` 是否为 Chen2020 |
| `KeyError: 'Terminal voltage'` | 变量名错误 | PyBaMM 变量名区分大小写，查文档 |


### 原理示意图 | Schematic Diagrams

- **battery_spm**: SPM单颗粒模型：正负极各用一个颗粒近似


In [ ]:
# [Bug修复 - 2026-10-25 给 Muse 说明]：
# 优先确保 labs 目录加入 sys.path，避免顺序单步执行本单元格时抛出 No module named 'common' 异常。
import sys, os
if '/home/jovyan/labs' not in sys.path:
    sys.path.insert(0, '/home/jovyan/labs')

# 原理图 | Principle Diagrams
from common.diagrams import *
import matplotlib.pyplot as plt

# SPM单颗粒模型：正负极各用一个颗粒近似
fig = battery_spm()
plt.show()


In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
import pybamm
pybamm.set_logging_level('ERROR')
from common.battery_adapter import BatteryFASTSimAdapter
import matplotlib.pyplot as plt
import numpy as np
print('✓ 导入成功')

In [ ]:
# 1C 放电
model = pybamm.lithium_ion.SPM()
param = pybamm.ParameterValues('Chen2020')
sim = pybamm.Simulation(model, parameter_values=param)
sim.solve([0, 3600])
sol = sim.solution

t = sol['Time [s]'].entries
V = sol['Terminal voltage [V]'].entries
cap = sol['Discharge capacity [A.h]'].entries

plt.figure(figsize=(10, 4))
plt.plot(cap, V)
plt.xlabel('Discharged capacity (Ah)'); plt.ylabel('Terminal voltage (V)')
plt.title('1C Discharge Curve (Chen2020 NMC, SPM)')
# 标注关键特征：中段电压平台与放电终点
mid = len(cap) // 2
plt.annotate(f'voltage plateau ~{V[mid]:.2f} V', xy=(cap[mid], V[mid]),
             xytext=(cap[mid] + 0.8, V[mid] + 0.28),
             arrowprops=dict(arrowstyle='->', color='#007aff'), color='#007aff')
plt.annotate(f'end: {cap[-1]:.2f} Ah, {V[-1]:.2f} V', xy=(cap[-1], V[-1]),
             xytext=(cap[-1] - 1.9, V[-1] + 0.4),
             arrowprops=dict(arrowstyle='->', color='#ff3b30'), color='#ff3b30')
plt.grid(True, alpha=0.3)
plt.show()

nominal_cap = cap[-1]
print(f'放出容量: {nominal_cap:.2f} Ah')

# 断言：容量合理（Chen2020 约 4.85Ah）
assert abs(nominal_cap - 4.85)/4.85 <= 0.05, '容量异常!'
print('✅ 评测通过！')

> 📊 **图意说明**：1C 恒流放电的电压-容量曲线。三段式形状是锂电芯的"指纹"：开头陡降（极化建立）→ 中段平台（标注点，大部分容量在此放出）→ 末端陡跌（活性锂耗尽，电压崩落）。曲线的横坐标终点读数就是放出容量（约 4.96 Ah）。后面每个 D 实验都是在解读这条曲线的不同侧面。


In [ ]:
# 导出 OCV-SOC CSV（演示用简化数据）
soc = np.linspace(0, 1, 20)
# 用放电曲线的电压近似 OCV（教学简化）
ocv = np.interp(soc, cap/cap[-1], V)

BatteryFASTSimAdapter.export_from_pybamm(
    soc, ocv, '/tmp/battery_pack_NMC_75Ah.csv',
    chem='NMC', capacity_ah=75.0)
print('✓ CSV 已导出: /tmp/battery_pack_NMC_75Ah.csv')

# 验证 Adapter 可加载
adapter = BatteryFASTSimAdapter('/tmp/battery_pack_NMC_75Ah.csv')
e_kwh = adapter.equivalent_energy_kwh(75.0)
print(f'等效能量: {e_kwh:.1f} kWh')

## 五、结果分析 | Results Analysis

### 5.1 容量验证 | Capacity Verification

实测放出容量 **4.96 Ah**，与标称 4.85 Ah 的偏差：
$$ \frac{4.96 - 4.85}{4.85} \times 100\% = +2.3\% < 5\% \quad ✅ $$

**解读**：
- Measured 4.96 Ah vs nominal 4.85 Ah: +2.3% deviation, within 5% tolerance.
- 正偏差 2.3% 属正常：厂家标称为保守值，留有余量
- SPM 模型精度验证通过，可用于后续 D2/D3 实验

### 5.2 电压曲线特征 | Voltage Curve Characteristics

典型恒流放电电压曲线三阶段：
| 阶段 | SOC 范围 | 电压特征 | 主导因素 |
|---|---|---|---|
| 初期陡降 | 100%→90% | 快速下降 0.1–0.2V | 欧姆极化 + 活化极化建立 |
| 中期平台 | 90%→20% | 平缓下降 | OCV 主导，极化稳定 |
| 末期陡降 | 20%→0% | 快速跌落至截止电压 | 浓差极化剧增 |

### 5.3 极化分析 | Polarization Analysis

端电压与开路电压的关系：
$$ V_{terminal} = OCV(SOC) - \eta_{ohm} - \eta_{act} - \eta_{conc} $$

- $\eta_{ohm} = I \cdot R_{int}$：欧姆极化，瞬时响应
- $\eta_{act}$：活化极化，Butler-Volmer 动力学
- $\eta_{conc}$：浓差极化，大电流/低 SOC 时主导

**教学点**：SPM 忽略了电解液浓差极化，这是 D4 必须用 DFN 的原因。

### 5.4 与理论对比

| 项目 | 理论/标称 | 实测 | 结论 |
|---|---|---|---|
| 容量 | 4.85 Ah | 4.96 Ah | ✅ +2.3% |
| 截止电压 | 2.5 V | 2.5 V | ✅ 一致 |
| 平均放电电压 | ~3.7 V | ~3.65 V | ✅ 吻合 |

### 5.5 工程意义

1. **容量标定**：BMS 的 SOC-OCV 查表基础
2. **一致性**：单体 2.3% 偏差，成组后需均衡
3. **导出接口**：本实验的 CSV 可直接供 C1 FASTSim 调用（见 D3）
### 5.6 设计改进思考 | Design Improvement Thinking

电池工程师拿到这条曲线后，怎么"读图改设计"？

1. **平台太短/容量不足**：指向活性材料装载量或材料本身的比容量——改配方（更高镍正极）或加厚涂布，但加厚会加重极化，看下一条。
2. **平台下倾严重（极化大）**：电极太厚、孔隙率太低或电解液电导不足——减薄电极、优化辊压密度、换低粘度电解液；代价是能量密度下降，这是电芯设计永恒的跷跷板。
3. **末端崩落太陡**：负极嵌锂动力学受限——减小石墨粒径或改用硅碳负极。
4. **仿真先行**：以上每一条改进都可以先在 PyBaMM 里改参数验证再开模做样品——D 系列四个实验（特性 D1 → 倍率 D2 → 标定 D3 → 快充安全 D4）正好对应电芯开发的完整体检表。


## 🏆 挑战任务
1. 对比 0.5C/1C/2C 放电曲线，解释倍率效应。
2. 把导出的 CSV 用 `adapter.apply_to_vehicle()` 映射到 FASTSim 车型，跑 UDDS 看电耗变化。

---

## 下一个实验 | Next →

**[D2_rate_capacity](./D2_rate_capacity.ipynb)**

> **所需内核**：`python3` | **Required kernel**: `python3`
